In [5]:
import torch

In [6]:
from datasets.indian_pines import IndianPinesDataset

ds = IndianPinesDataset('data/indian_pines/Indian_pines_corrected.mat','data/indian_pines/Indian_pines_gt.mat')
data, labels = ds[0]
print(data.shape)    # (200, 145, 145)
print(labels.shape)  # (145, 145)
print(labels.max())  # should be 16
print(labels.min())  # should be 0 (unlabeled pixels)

torch.Size([200, 145, 145])
torch.Size([145, 145])
tensor(16)
tensor(0)


In [7]:
from preprocessing.continuum_removal import ContinuumRemoval
layer = ContinuumRemoval()
dummy = torch.randn(4, 200, 11, 11)  # batch of 4 patches
out = layer(dummy)
print(out.shape)  # should be torch.Size([4, 200, 11, 11])

torch.Size([4, 200, 11, 11])


In [8]:
import scipy.io
data = scipy.io.loadmat('data/indian_pines/Indian_pines_corrected.mat')
print(data.keys())

gt = scipy.io.loadmat('data/indian_pines/Indian_pines_gt.mat')
print(gt.keys())

dict_keys(['__header__', '__version__', '__globals__', 'indian_pines_corrected'])
dict_keys(['__header__', '__version__', '__globals__', 'indian_pines_gt'])


In [9]:
from models.spectral_3d_block import Spectral3DStack
stack = Spectral3DStack(num_bands=200, num_filters=8, num_blocks=3)
dummy = torch.randn(1, 200, 145, 145)
out = stack(dummy)
print(out.shape)  # (1, 1600, 145, 145)
print(f"Out channels: {stack.out_channels}")  # 1600

torch.Size([1, 1600, 145, 145])
Out channels: 1600


In [10]:
from models.se_block import SEBlock

se = SEBlock(channels=3200)
dummy = torch.randn(1, 3200, 145, 145)
out = se(dummy)
print(out.shape)  # (1, 3200, 145, 145)

torch.Size([1, 3200, 145, 145])


In [11]:
from models.encoder_2d import Encoder2D

enc = Encoder2D(in_channels=3200, base_filters=32)
dummy = torch.randn(1, 3200, 145, 145)
b, skips = enc(dummy)
print(b.shape)                    # (1, 512, 9, 9)
for s in skips:
    print(s.shape)
# (1, 32,  145, 145)
# (1, 64,  72,  72)
# (1, 128, 36,  36)
# (1, 256, 18,  18)

torch.Size([1, 512, 9, 9])
torch.Size([1, 32, 145, 145])
torch.Size([1, 64, 72, 72])
torch.Size([1, 128, 36, 36])
torch.Size([1, 256, 18, 18])


In [12]:
from models.decoder_2d import Decoder2D

dec = Decoder2D(num_classes=17, base_filters=32)
skips = [
    torch.randn(1, 32, 145, 145),
    torch.randn(1, 64, 72, 72),
    torch.randn(1, 128, 36, 36),
    torch.randn(1, 256, 18, 18),
]
b = torch.randn(1, 512, 9, 9)
out = dec(b, skips)
print(out.shape)  # (1, 17, 144, 144)

torch.Size([1, 17, 145, 145])


In [14]:
from models.hyperspectral_net import HyperspectralNet

model = HyperspectralNet(num_bands=200, num_classes=17)
dummy = torch.randn(1, 200, 145, 145)
out = model(dummy)
print(out.shape)  # (1, 17, 145, 145)

torch.Size([1, 17, 145, 145])
